# Qwen3.5-9B — SFT marketing với Unsloth (LoRA)

Notebook tham khảo [`notebooks/08_Qwen25_05B_Marketing_Unsloth_Base_vs_Instruct.ipynb`](../notebooks/08_Qwen25_05B_Marketing_Unsloth_Base_vs_Instruct.ipynb): cùng định dạng Alpaca, dữ liệu từ [`dataset/marketing/marketing_social_media_dataset.json`](../dataset/marketing/marketing_social_media_dataset.json) trong repo.

**Mô hình mặc định:** [`Qwen/Qwen3.5-9B`](https://huggingface.co/Qwen/Qwen3.5-9B) (bản post-trained / chat). Nếu muốn fine-tune từ pretrain thuần, đổi `MODEL_ID` thành [`Qwen/Qwen3.5-9B-Base`](https://huggingface.co/Qwen/Qwen3.5-9B-Base).

**Gợi ý Unsloth nhanh hơn / 4-bit có sẵn:** có thể đặt `MODEL_ID = "unsloth/Qwen3.5-9B"` (mirror tối ưu) thay cho repo `Qwen/` nếu môi trường của bạn tương thích.

**VRAM:** 9B + 4-bit + LoRA thường cần **khoảng 16–24 GB** tùa `MAX_SEQ_LENGTH` và batch. Nếu OOM, hạ `PER_DEVICE_BATCH`, tăng `GRAD_ACCUM`, hoặc giảm `MAX_SEQ_LENGTH` (ví dụ 1024).

**Luồng:** cài Unsloth → chuẩn hoá `text` theo template Alpaca → LoRA SFT → **merge + xuất GGUF** (mục 5) → so sánh sinh văn **trước / sau** fine-tune.

**LM Studio:** sau khi có file GGUF, cách import và bật server mô tả trong [README.md](../README.md#lm-studio-importing-fine-tuned-models).

## 1. Cấu hình và đường dẫn

In [15]:
import os

# RTX 5060 Ti (Blackwell/sm_120): ngăn TF và xformers load CUDA DLL gây crash trên Windows
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("XFORMERS_DISABLED", "1")
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported

import gc
import platform
import random
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "notebooks").is_dir() and REPO_ROOT.name != "notebooks":
    pass
elif (REPO_ROOT.parent / "notebooks").is_dir():
    REPO_ROOT = REPO_ROOT.parent


SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
DTYPE = None

# Repo chính thức Qwen (theo yêu cầu). Thay bằng "unsloth/Qwen3.5-9B" nếu muốn bản Unsloth tối ưu.
MODEL_ID = "Qwen/Qwen3.5-2B"
# Dùng replace để tránh "/" trong MODEL_ID tạo subdir ngoài ý muốn khi nối Path
_model_slug = MODEL_ID.replace("/", "-")
MODELS_DIR = REPO_ROOT / "mcs_train_content_model_outputs" / f"{MODEL_ID}_facebook_content_SFT"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LORA_OUTPUT_DIR = MODELS_DIR / f"lora_{_model_slug}_facebook_content_SFT"

# Alpaca JSON trong repo (cùng schema instruction / input / response)
DATASET_JSON = REPO_ROOT / "dataset" / "fnb_dataset_vi.json"
if not DATASET_JSON.is_file():
    _alt = REPO_ROOT.parent / "dataset" / "fnb_dataset_vi.json"
    if _alt.is_file():
        DATASET_JSON = _alt
assert DATASET_JSON.is_file(), f"Không tìm thấy file dataset: {DATASET_JSON}"

VAL_FRACTION = 0.08
NUM_TRAIN_EPOCHS = 2
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 16
LEARNING_RATE = 2e-4

DATASET_NUM_PROC = None if platform.system() == "Windows" else 2

assert torch.cuda.is_available(), "Cần GPU + CUDA để chạy Unsloth 4-bit hợp lý."
print("CUDA:", torch.cuda.get_device_name(0))
print("MODEL_ID:", MODEL_ID)
print("MODELS_DIR:", MODELS_DIR)
print("LORA_OUTPUT_DIR:", LORA_OUTPUT_DIR)
print("DATASET_JSON:", DATASET_JSON)

CUDA: NVIDIA GB10
MODEL_ID: Qwen/Qwen3.5-2B
MODELS_DIR: /home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen/Qwen3.5-2B_facebook_content_SFT
LORA_OUTPUT_DIR: /home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen/Qwen3.5-2B_facebook_content_SFT/lora_Qwen-Qwen3.5-2B_facebook_content_SFT
DATASET_JSON: /home/dc34rpa/nathan/dataset/fnb_dataset_vi.json


## 2. Dataset và template Alpaca

Ghép `instruction` + `input` + `response` và nối **EOS** để tránh sinh không dừng.

In [16]:
raw = load_dataset("json", data_files=str(DATASET_JSON))
if "train" in raw:
    full = raw["train"]
else:
    full = raw[list(raw.keys())[0]]

full = full.shuffle(seed=SEED)
n_val = max(1, int(len(full) * VAL_FRACTION))
eval_ds = full.select(range(n_val))
train_ds = full.select(range(n_val, len(full)))
print("train:", len(train_ds), "eval:", len(eval_ds), "columns:", train_ds.column_names)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


def add_text_column(batch, eos_token: str):
    texts = []
    for ins, inp, out in zip(batch["instruction"], batch["input"], batch["response"]):
        inp = inp if inp is not None else ""
        texts.append(alpaca_prompt.format(ins, inp, out) + eos_token)
    return {"text": texts}


_tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if _tok.pad_token is None:
    _tok.pad_token = _tok.eos_token
EOS_TOKEN = _tok.eos_token
print("EOS_TOKEN repr:", repr(EOS_TOKEN[:40] if len(EOS_TOKEN) > 40 else EOS_TOKEN))

train_tok = train_ds.map(
    lambda b: add_text_column(b, EOS_TOKEN),
    batched=True,
    num_proc=DATASET_NUM_PROC,
    remove_columns=train_ds.column_names,
)
eval_tok = eval_ds.map(
    lambda b: add_text_column(b, EOS_TOKEN),
    batched=True,
    num_proc=DATASET_NUM_PROC,
    remove_columns=eval_ds.column_names,
)
del _tok
gc.collect()

Generating train split: 0 examples [00:00, ? examples/s]

train: 460 eval: 40 columns: ['instruction', 'input', 'response']
EOS_TOKEN repr: '<|im_end|>'


Map (num_proc=2):   0%|          | 0/460 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/40 [00:00<?, ? examples/s]

7705

## 3. Huấn luyện LoRA (Unsloth + TRL `SFTTrainer`)

In [17]:
def train_lora(model_name: str, output_dir: Path, train_dataset):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )

    args = SFTConfig(
        output_dir=str(output_dir / "trainer_output"),
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.1,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to="none",
        save_strategy="no",
        padding_free=False,
        max_length=MAX_SEQ_LENGTH,
    )

    _sft_kw = dict(
        model=model,
        train_dataset=train_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=DATASET_NUM_PROC,
        packing=False,
        args=args,
    )
    try:
        trainer = SFTTrainer(tokenizer=tokenizer, **_sft_kw)
    except TypeError:
        trainer = SFTTrainer(processing_class=tokenizer, **_sft_kw)

    trainer.train()
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    stats = trainer.state.log_history

    del trainer, model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return stats

## 4. Chạy fine-tune

Lưu adapter + tokenizer tại `LORA_OUTPUT_DIR`.

In [18]:
print("=== Train LoRA ===")
_ = train_lora(MODEL_ID, LORA_OUTPUT_DIR, train_tok)
print("Saved:", LORA_OUTPUT_DIR)

=== Train LoRA ===
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.676 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.1. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

/home/dc34rpa/nathan/myenv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/460 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 460 | Num Epochs = 2 | Total steps = 58
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.323964
10,1.945440
15,1.676840
20,1.483499
25,1.409253
30,1.347193
35,1.271637
40,1.224310
45,1.223762
50,1.195642


Saved: /home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen/Qwen3.5-2B_facebook_content_SFT/lora_Qwen-Qwen3.5-2B_facebook_content_SFT


## 5. Merge LoRA và xuất (HF 16-bit + GGUF / LM Studio & Ollama)

Chạy sau khi đã train xong và có thư mục `LORA_OUTPUT_DIR`.

- Nạp lại adapter **không** dùng 4-bit, gộp vào base, rồi lưu bản **merged 16-bit** (thư mục `merged_16bit_marketing`) cho Transformers / vLLM — **đây là bản gốc (không quant Q4/Q8) trên stack HF**.
- Xuất **GGUF** vào `gguf` cho **LM Studio** / **Ollama**. Mặc định dùng **`f16` trong GGUF** (trọng số 16-bit, không dùng quant kiểu `q4_k_m`). Đổi `GGUF_QUANT` sang `q4_k_m` (hoặc `q5_k_m`, `q8_0`) nếu cần file nhỏ hơn / vừa VRAM; file `f16` rất lớn với 9B.

Nếu thiếu VRAM khi merge FP16, đặt `SAVE_MERGED_16BIT = False` — vẫn chạy bước xuất GGUF sau đó. Giảm kích thước file GGUF: đổi `GGUF_QUANT` từ `f16` sang `q4_k_m` (hoặc tương tự).

Hướng dẫn import và server: [README.md](../README.md#lm-studio-importing-fine-tuned-models).

In [19]:
from pathlib import Path
import gc
import torch

assert LORA_OUTPUT_DIR.exists(), f"Chưa có LoRA, hãy chạy ô train trước: {LORA_OUTPUT_DIR}"

MERGED_OUTPUT_DIR = MODELS_DIR / "merged_16bit"
GGUF_OUTPUT_DIR   = MODELS_DIR / "gguf"
MERGED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_MERGED_16BIT = True
GGUF_QUANT = "f16"

print("Loading LoRA for merge / GGUF...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)

if SAVE_MERGED_16BIT:
    print("Saving merged 16-bit →", MERGED_OUTPUT_DIR)
    model.save_pretrained_merged(str(MERGED_OUTPUT_DIR), tokenizer, save_method="merged_16bit")

print("Exporting GGUF →", GGUF_OUTPUT_DIR, "| quant =", GGUF_QUANT)
model.save_pretrained_gguf(str(GGUF_OUTPUT_DIR), tokenizer, quantization_method=GGUF_QUANT)

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Xong. GGUF:", GGUF_OUTPUT_DIR)

Loading LoRA for merge / GGUF...
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.676 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.1. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Saving merged 16-bit → /home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen/Qwen3.5-2B_facebook_content_SFT/merged_16bit
Found HuggingFace hub cache directory: /home/dc34rpa/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


...

Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen/Qwen3.5-2B_facebook_content_SFT/gguf`: 100%|█████████████████████| 1/1 [00:01<00:00,  1.46s/it]


Successfully copied all 1 files from cache to `/home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen/Qwen3.5-2B_facebook_content_SFT/gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:18<00:00, 18.41s/it]


Unsloth: Merge process complete. Saved to `/home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen/Qwen3.5-2B_facebook_content_SFT/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen/Qwen3.5-2B_facebook_content_SFT/gguf_gguf/Qwen3.5-2B.BF16.gguf', '/home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen/Qwen3.5-2B_facebook_content_SFT/gguf_gguf/Qwen3.5-2B.BF16-mmproj.gguf']
U

## 6. So sánh sinh văn: pretrained vs sau SFT

Đổi `EXAMPLE_IDX` để thử mẫu khác trong `eval_ds`.

In [20]:
# ============================================================
# CELL 6 — So sánh sinh văn (dùng transformers thuần, không Unsloth)
# ============================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc

def build_prompt_no_response(instruction: str, input_text: str) -> str:
    input_text = input_text or ""
    return alpaca_prompt.format(instruction, input_text, "")

@torch.inference_mode()
def generate_one(model, tokenizer, prompt: str, max_new_tokens: int = 256):
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_attention_mask=True,
    )
    input_ids      = encoded.input_ids.to(model.device)
    attention_mask = encoded.attention_mask.to(model.device)

    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

    # Chỉ decode phần sinh ra, bỏ qua prompt
    generated_ids = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def load_model_for_inference(model_path: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # AutoModelForCausalLM với trust_remote_code load được cả VLM text-only mode
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    return model, tokenizer


COMPARE_VARIANTS = [
    ("Pretrained (chưa SFT marketing)", MODEL_ID, False),
    ("+ LoRA marketing", str(LORA_OUTPUT_DIR), True),
]

EXAMPLE_IDX = 0
row = eval_ds[EXAMPLE_IDX]
prompt = build_prompt_no_response(row["instruction"], row.get("input") or "")

print("=" * 80)
print("INSTRUCTION:\n", row["instruction"])
print("\nINPUT:\n", row.get("input") or "(none)")
print("\nGOLD RESPONSE:\n", str(row["response"]))
print("=" * 80)

for name, model_path, _is_adapter in COMPARE_VARIANTS:
    print(f"\nLoading {name} ...")
    model, tokenizer = load_model_for_inference(model_path)

    gen = generate_one(model, tokenizer, prompt)
    print(f"\n{'─' * 40}\n>>> {name}\n{'─' * 40}\n{gen}\n")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

INSTRUCTION:
 Viết caption Facebook cho thương hiệu thực phẩm đông lạnh tiện lợi, định vị là giải pháp bữa ăn nhanh – ngon – sạch cho dân văn phòng và gia đình trẻ tại Việt Nam. Bài viết cần nhấn mạnh offer rõ ràng, tạo nhu cầu mua ngay, tối ưu cho mục tiêu chuyển đổi đơn hàng trên Facebook với KPI cụ thể về CTR, CPL và GMV.

INPUT:
 Công ty: An Tâm Food Việt Nam
Đối tượng: Nhân viên văn phòng 25-35 tuổi, mẹ bỉm và gia đình trẻ ở TP.HCM, Hà Nội
Sản phẩm/dịch vụ: Combo thực phẩm đông lạnh tiện lợi gồm há cảo tôm, chả giò, viên thả lẩu, gà ướp sốt sẵn, giao nhanh trong ngày
Giới hạn/Ngân sách: 35 triệu VND cho chiến dịch Facebook tháng này
Offer/ưu đãi: Giảm 20% cho combo đầu tiên, freeship đơn từ 299.000 VND, tặng thêm 1 gói sốt chấm khi đặt qua inbox
Mục tiêu KPI cụ thể: CTR tối thiểu 2,8%, CPL dưới 25.000 VND, GMV đạt 180 triệu VND, AOV trên 320.000 VND
Giai đoạn workflow: Caption Facebook
Kênh Facebook: Fanpage chính thức + quảng cáo chuyển đổi + inbox Messenger

GOLD RESPONSE:
 Tan 

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


────────────────────────────────────────
>>> Pretrained (chưa SFT marketing)
────────────────────────────────────────
Dưới đây là 3 mẫu caption Facebook được tối ưu hóa cho chiến dịch của **An Tâm Food Việt Nam**, được thiết kế riêng để đạt được các KPI: **CTR ≥ 2,8%, CPL < 25.000 VND, GMV 180 triệu VND** và **AOV > 320.000 VND**.

Các mẫu này được chia theo 3 chiến lược khác nhau để bạn có thể A/B test và điều chỉnh ngân sách:

---

### Mẫu 1: Tập trung vào "Hết đói nhanh" (Tối ưu CTR & AOV)
*Chiến lược: Nhấn mạnh sự tiện lợi và giải pháp bữa ăn nhanh cho văn phòng, tạo cảm giác cấp thiết.*

**Caption:**
🚀 **HẾT ĐÓI NGAY! Bạn đang làm việc quá muộn nhưng chưa ăn gì?**
🍲 **Combo An Tâm Food Việt Nam** – Giải pháp bữa ăn nhanh ngon, sạch, tiện lợi cho văn phòng và gia đình trẻ.

✅ **Đừng để bụng đói khiến bạn làm việc kém hiệu quả!**
🔥 **GIẢM 20% ĐỐI VỚI COMBO ĐẦU TIÊN


Loading + LoRA marketing ...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]


────────────────────────────────────────
>>> + LoRA marketing
────────────────────────────────────────
Bạn cần bữa ăn nhanh, ngon, sạch và tiện hơn trong ngày?

An Tâm Food Việt Nam ra mắt combo thực phẩm đông lạnh tiện lợi dành cho dân văn phòng và gia đình trẻ: há cảo tôm, chả giò, viên thả lẩu, gà ướp sốt sẵn.

Ưu đãi tháng này rất rõ ràng: giảm 20% cho combo đầu tiên, freeship đơn từ 299.000 VND và tặng thêm 1 gói sốt chấm khi đặt qua inbox.

Nếu bạn đang tìm giải pháp bữa ăn nhanh nhưng vẫn muốn giữ chất lượng, đây là lựa chọn hợp lý.

Inbox ngay để nhận bảng giá và đặt combo hôm nay!

KPI chiến dịch: CTR 2,8%+, CPL dưới 25.000 VND, GMV 180 triệu VND, AOV 320.000 VND



## 7. Prompt tùy chọn (ví dụ F&B tiếng Việt)

In [21]:
import torch
import gc

@torch.inference_mode()
def generate_one(model, tokenizer, prompt: str, max_new_tokens: int = 256):
    messages = [{"role": "user", "content": prompt}]
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        text = prompt

    # Qwen3VLProcessor wraps a tokenizer internally — access it directly
    tok = getattr(tokenizer, "tokenizer", tokenizer)
    input_ids = tok.encode(text, return_tensors="pt").to(model.device)
    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
    decoded = tok.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True,
    )
    return decoded


# ── Inference ──────────────────────────────────────────────────────────────────

CUSTOM_INSTRUCTION = (
    "Xây dựng chiến dịch email marketing có mục tiêu để quảng bá cho buổi ra mắt sản phẩm mới."
)
CUSTOM_INPUT = """\
Company: Cà phê Rang Xay Sài Gòn — thương hiệu cà phê đặc sản nội địa, chuyên dòng single-origin Tây Nguyên
Target Audience: Chủ quán cà phê, F&B manager, người yêu cà phê chuyên nghiệp (25–45 tuổi) tại TP.HCM và Hà Nội
Constraints: Tỷ lệ mở email cao (~30%) nhưng tỷ lệ chuyển đổi thấp (~3%)
Goals: Đạt tỷ lệ mở 35%, tỷ lệ chuyển đổi 12% trong 2 tuần ra mắt
Workflow Stage: Copywriting"""

custom_prompt = build_prompt_no_response(CUSTOM_INSTRUCTION, CUSTOM_INPUT)
print("=" * 80)
print("INSTRUCTION:\n", CUSTOM_INSTRUCTION)
print("\nINPUT:\n", CUSTOM_INPUT)
print("=" * 80)

for name, model_path, _ in COMPARE_VARIANTS:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    FastLanguageModel.for_inference(model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    gen = generate_one(model, tokenizer, custom_prompt, max_new_tokens=400)
    print(f"\n{'─' * 40}\n>>> {name}\n{'─' * 40}\n{gen}\n")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

INSTRUCTION:
 Xây dựng chiến dịch email marketing có mục tiêu để quảng bá cho buổi ra mắt sản phẩm mới.

INPUT:
 Company: Cà phê Rang Xay Sài Gòn — thương hiệu cà phê đặc sản nội địa, chuyên dòng single-origin Tây Nguyên
Target Audience: Chủ quán cà phê, F&B manager, người yêu cà phê chuyên nghiệp (25–45 tuổi) tại TP.HCM và Hà Nội
Constraints: Tỷ lệ mở email cao (~30%) nhưng tỷ lệ chuyển đổi thấp (~3%)
Goals: Đạt tỷ lệ mở 35%, tỷ lệ chuyển đổi 12% trong 2 tuần ra mắt
Workflow Stage: Copywriting
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.676 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.1. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

/home/dc34rpa/nathan/myenv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/dc34rpa/nathan/myenv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



────────────────────────────────────────
>>> Pretrained (chưa SFT marketing)
────────────────────────────────────────
Chào bạn, với chiến dịch email marketing cho **Cà phê Rang Xay Sài Gòn** (đặc sản Tây Nguyên) nhắm vào đối tượng F&B và chủ quán (25–45 tuổi), việc đạt được tỷ lệ mở 35% và chuyển đổi 12% trong 2 tuần là thách thức lớn do tỷ lệ mở hiện tại chỉ là 30%.

Dưới đây là chiến lược Copywriting chi tiết để tối ưu hóa hiệu quả:

### 1. Phân tích Pain Point & Hook (Góc nhìn người đọc)
Đối tượng F&B chủ yếu là người bận rộn, quan tâm đến lợi nhuận và chất lượng.
*   **Hook 1 (Dữ liệu):** "Bạn đang bỏ phí hàng triệu đồng trên mỗi đơn hàng vì không có sản phẩm đặc sản?"
*   **Hook 2 (Thách thức):** "Tỷ lệ mở email của bạn chỉ là 30% – có nghĩa là 70% khách hàng đang bỏ qua tin nhắn của bạn. Chúng tôi đã làm gì?"
*   **Hook 3 (Giải pháp):** "Giải pháp 'Cà phê Rang Xay Sài Gòn' – Chỉ 15 phút để tìm ra nguồn cà phê Tây Nguyên chất lượng nhất."

### 2. Cấu trúc Email (3 Phần)

#### Phầ

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]


────────────────────────────────────────
>>> + LoRA marketing
────────────────────────────────────────
Subject: Ra mắt cà phê Rang Xay Sài Gòn:品味 Tây Nguyên, sống mỗi ngày

Chào bạn,

Nếu bạn đang tìm một ly cà phê không chỉ ngon, mà còn mang đến một trải nghiệm “đúng vị” cho buổi sáng hay giờ làm việc, thì đây là lúc bạn cần gặp Rang Xay Sài Gòn.

Chúng tôi không chỉ bán cà phê. Chúng tôi mang đến một dòng sản phẩm single-origin Tây Nguyên, được tuyển chọn kỹ lưỡng từ các vùng đất có khí hậu đặc trưng, để bạn thưởng thức được sự kết hợp giữa vị đất, nắng và tinh thần làm việc.

Buổi ra mắt của chúng tôi sẽ khai mạc tại 12 điểm đặt hàng tại TP.HCM và Hà Nội trong 2 tuần tới. Mục tiêu của chúng tôi rất rõ: tăng tỷ lệ mở email lên 35% và kéo tỷ lệ chuyển đổi từ 12% trong giai đoạn ra mắt.

Nếu bạn đang cần một ly cà phê để tỉnh táo, nâng cao năng lượng hoặc nâng tầm trải nghiệm F&B, đây là thời điểm để thử ngay.

Inbox ngay để nhận ưu đãi ra mắt và đặt bàn.

Rang Xay Sài Gòn
Cà phê đặc 

### Ghi chú

- **Qwen3.5-9B** là mô hình đa phương thức; notebook này chỉ dùng nhánh **văn bản** qua template Alpaca (giống notebook 0.5B).
- **Merge / GGUF:** mặc định `GGUF_QUANT = "f16"` (không quant Q4). Nếu thiếu ổ đĩa/VRAM, đổi sang `q4_k_m` hoặc tương tự. Nếu `save_pretrained_gguf` lỗi, cập nhật Unsloth; có thể bỏ `save_pretrained_merged` (`SAVE_MERGED_16BIT = False`) rồi thử lại. Import: [README.md](../README.md#lm-studio-importing-fine-tuned-models).
- Nếu Unsloth báo lỗi kiến trúc / thiếu patch, cập nhật `unsloth` và `transformers` theo [tài liệu Unsloth](https://github.com/unslothai/unsloth).
- Windows: xem [hướng dẫn cài Unsloth](https://unsloth.ai/docs/get-started/install/windows-installation).